# TinyDoc-VLM: Colab Training

Trains all 3 stages on T4 (free tier). Auto-resumes from Google Drive checkpoints.

**IMPORTANT**:
- Runtime → Change runtime type → **T4 GPU**
- Run cells in order. If disconnected, re-run from the top — auto-resume picks up where you left off.

## 1. Mount Google Drive

All checkpoints and data live here. Only ~2GB total.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print(f'Drive ready at {DRIVE_ROOT}')

## 2. Clone repo & install deps

In [ ]:
import os, sys
REPO_DIR = '/content/tinydoc-vlm'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/eulogik/TinyDoc-VLM.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers sentencepiece tokenizers pillow numpy pandas tqdm pyyaml gradio optimum onnx onnxruntime einops faker jinja2 pydantic
!pip install -q accelerate
print('Deps installed')

## 3. Prepare data

Copies from Drive if already generated, otherwise generates 10K synthetic docs.

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = 'data/synthetic'
DRIVE_DATA = f'{DRIVE_ROOT}/data/synthetic'

if os.path.exists(f'{DRIVE_DATA}/output/manifest.jsonl'):
    print('Copying data from Drive...')
    shutil.copytree(DRIVE_DATA, DATA_DIR, dirs_exist_ok=True)
    print(f'Done. Manifest: {sum(1 for _ in open(f"{DATA_DIR}/output/manifest.jsonl"))} docs')
else:
    print('Generating 10K synthetic documents (~15 min)...')
    !python -c "from data.synthetic.generator import generate_dataset; generate_dataset(num_docs=10000, output_dir='data/synthetic/output')"
    print('Copying data to Drive for future sessions...')
    shutil.copytree(DATA_DIR, DRIVE_DATA, dirs_exist_ok=True)

## 4. Select training stage

Stage 1 → Stage 2 → Stage 3 sequentially. Each auto-resumes.

In [ ]:
# @title Stage selection
STAGE = 1 # @param {type:"slider", min:1, max:3, step:1}

print(f'Selected Stage {STAGE}')
print(f'  Stage 1: Layout pretraining (~15 min on T4)')
print(f'  Stage 2: Doc understanding (~45 min on T4)')
print(f'  Stage 3: Instruction tuning (~30 min on T4)')

## 5. Build T4-optimized config

Scales down from the 8×A100 configs to fit T4 16GB.

In [ ]:
import yaml

def build_t4_config(stage: int) -> dict:
    base = {
        1: 'training/stage1_layout_pretrain.yaml',
        2: 'training/stage2_doc_understanding.yaml',
        3: 'training/stage3_instruction_tuning.yaml',
    }[stage]

    with open(base) as f:
        cfg = yaml.safe_load(f)

    # T4-friendly overrides: smaller batch, more accumulation
    cfg['training']['batch_size'] = 4
    cfg['training']['gradient_accumulation_steps'] = 8
    cfg['training']['save_every_steps'] = 500
    cfg['training']['eval_every_steps'] = 250
    cfg['training']['log_every_steps'] = 10
    cfg['training']['gradient_checkpointing'] = True
    cfg['training']['num_epochs'] = 1  # one epoch per session

    # Stage 2/3 need pretrained model from previous stage
    if stage > 1:
        prev_stage = stage - 1
        cfg['model']['pretrained_checkpoint'] = f'{DRIVE_ROOT}/checkpoints/stage{prev_stage}/best'

    # Point output to Drive for persistence
    cfg['training']['output_dir'] = f'{DRIVE_ROOT}/checkpoints/stage{stage}'

    return cfg

cfg = build_t4_config(STAGE)
print(yaml.dump(cfg, default_flow_style=False))

## 6. Run training (with auto-resume)

Checks Drive for existing Stage `{STAGE}` checkpoint and resumes. Syncs back to Drive after each save.

In [ ]:
import torch
import logging
from torch.utils.data import random_split
from pathlib import Path

from tinydoc_vlm import (
    TinyDocVLMConfig,
    TinyDocVLMForConditionalGeneration,
    TinyDocVLMProcessor,
    TinyDocVLMTrainer,
    TrainerConfig,
    DocumentDataset,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('colab')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {device}')

# --- Build model ---
mc = cfg['model']
config = TinyDocVLMConfig(
    vision_config=mc.get('vision_config'),
    decoder_config=mc.get('decoder_config'),
    pixel_shuffle_scale=mc.get('pixel_shuffle_scale', 3),
    image_size=mc.get('image_size', 384),
    patch_size=mc.get('patch_size', 16),
)
model = TinyDocVLMForConditionalGeneration(config)

# Check for pretrained from previous stage
pretrained = mc.get('pretrained_checkpoint')
if pretrained and os.path.exists(pretrained):
    logger.info(f'Loading pretrained checkpoint: {pretrained}')
    model = TinyDocVLMForConditionalGeneration.from_pretrained(pretrained)

# --- Build dataset ---
dc = cfg['data']
dataset = DocumentDataset(
    data_root=dc.get('data_root', 'data/synthetic'),
    manifest_path=dc.get('manifest_path'),
    max_seq_length=dc.get('max_seq_length', 2048),
    stage=dc.get('stage', 1),
)

train_size = int(0.9 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])
logger.info(f'Dataset: {len(dataset)} total ({train_size} train / {eval_size} eval)')

# --- Build trainer config ---
tc = cfg['training']
trainer_cfg = TrainerConfig(
    output_dir=tc.get('output_dir', 'checkpoints'),
    num_epochs=tc.get('num_epochs', 1),
    batch_size=tc.get('batch_size', 4),
    gradient_accumulation_steps=tc.get('gradient_accumulation_steps', 8),
    learning_rate=tc.get('learning_rate', 1e-4),
    min_learning_rate=tc.get('min_learning_rate', 1e-5),
    warmup_steps=tc.get('warmup_steps', 500),
    weight_decay=tc.get('weight_decay', 0.01),
    max_grad_norm=tc.get('max_grad_norm', 1.0),
    max_seq_length=tc.get('max_seq_length', 2048),
    stage=tc.get('stage', 1),
    use_fp16=tc.get('use_fp16', True),
    save_every_steps=tc.get('save_every_steps', 500),
    eval_every_steps=tc.get('eval_every_steps', 250),
    log_every_steps=tc.get('log_every_steps', 10),
    gradient_checkpointing=tc.get('gradient_checkpointing', True),
)

# --- Create trainer ---
processor = TinyDocVLMProcessor()
trainer = TinyDocVLMTrainer(
    model=model,
    processor=processor,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    config=trainer_cfg,
    device=device,
)

# --- Auto-resume: check for existing checkpoint in Drive ---
checkpoint_dir = Path(tc['output_dir'])
if (checkpoint_dir / 'trainer_state.pt').exists():
    import glob
    ckpts = sorted((checkpoint_dir).glob('step_*'))
    if ckpts:
        latest = str(ckpts[-1])
        logger.info(f'Resuming from checkpoint: {latest}')
        trainer.load_checkpoint(latest)
    else:
        logger.info('No step checkpoint found, starting fresh')
else:
    logger.info('No existing checkpoint, starting fresh')

trainer.train()
logger.info(f'Stage {STAGE} training complete!')

## 7. Sync to Drive

Final sync to ensure latest checkpoint is in Drive for next stage.

In [ ]:
import shutil

src = Path(cfg['training']['output_dir'])
dst = Path(DRIVE_ROOT) / 'checkpoints' / f'stage{STAGE}'
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f'Synced to {dst}')
print('\nDone! Ready to run Stage', STAGE + 1, 'or download the final model.')

## 8. Download final model (optional)

After all 3 stages complete, download the final model for local inference.

In [ ]:
# After Stage 3 is done, download
from google.colab import files

final_dir = f'{DRIVE_ROOT}/checkpoints/stage3/best'
if os.path.exists(final_dir):
    !zip -r /content/tinydoc-vlm-final.zip {final_dir}
    files.download('/content/tinydoc-vlm-final.zip')
else:
    print('Stage 3 not complete yet. Run all 3 stages first.')